# 01 — Исследование датасета ForteBank

Цель этой тетради: понять структуру открытой выборки **до обучения модели**. Источник требований — кейс ForteBank (`Кейс_ForteBank.docx`): не меньше 40 роликов по 10–145 секунд, у каждого есть флаг нарушения и таймкод, классы организаторы не раскрывают.

**Видео читаются из папки `open/`.** Ноутбук ищет ролики в таком порядке: `open/`, `data/open/`, `data/raw/open/`, затем рекурсивно внутри найденной папки. Формат кейса — `.mov`, остальные контейнеры тоже подхватываются.

Разметка команды: `docs/video-labels-split.md` и `docs/video-label-schema.md`. Ссылки на Правила НБ РК №231, Правила №14 и Закон №191-IV — гипотезы, не юридическое заключение. Необычное действие нельзя автоматически считать нарушением.

После прогона сверху вниз:

1. Таблица технических параметров всех видео из `open/`.
2. Распределения длительности, FPS, разрешения и меток.
3. Таблица `data/annotations/events.csv` с категориями и уверенностью.
4. Кадры за 4 секунды до таймкода и 4 секунды после.
5. Заготовка зон кадра (координаты — только после просмотра).


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib.patches import Rectangle
from tqdm.auto import tqdm

try:
    import cv2
except ImportError:
    cv2 = None

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 50)

VIDEO_EXTENSIONS = {".mov", ".mp4", ".avi", ".mkv", ".m4v"}
WINDOW_SEC = 4.0
FRAME_STEP_SEC = 2.0


def project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() or (candidate / "notebooks").exists() or (candidate / "open").exists():
            return candidate
    return start if start.name != "notebooks" else start.parent


def has_videos(directory: Path) -> bool:
    if not directory.is_dir():
        return False
    return any(path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS for path in directory.rglob("*"))


def resolve_open_dir(root: Path) -> Path:
    env = os.environ.get("VIDEO_DIR") or os.environ.get("FORTEBANK_OPEN_DIR")
    candidates = []
    if env:
        candidates.append(Path(env).expanduser().resolve())
    candidates.extend([
        root / "open",
        root / "data" / "open",
        root / "data" / "raw" / "open",
        root.parent / "open",
        Path.cwd() / "open",
        root / "data" / "raw",
    ])
    unique = []
    seen = set()
    for path in candidates:
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        unique.append(path)
    for path in unique:
        if has_videos(path):
            return path
    for path in unique:
        if path.is_dir():
            return path
    default = root / "open"
    default.mkdir(parents=True, exist_ok=True)
    return default


cwd = Path.cwd().resolve()
ROOT = project_root(cwd)
DATA_DIR = resolve_open_dir(ROOT)
ANNOTATIONS_DIR = ROOT / "data" / "annotations"
PROCESSED_DIR = ROOT / "data" / "processed"
FIGURES_DIR = ROOT / "reports" / "figures"
EVENT_FRAMES_DIR = FIGURES_DIR / "event_frames"
EVENTS_PATH = ANNOTATIONS_DIR / "events.csv"

for path in (ANNOTATIONS_DIR, PROCESSED_DIR, FIGURES_DIR, EVENT_FRAMES_DIR, DATA_DIR):
    path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Open videos:  {DATA_DIR}")
print(f"Папка существует: {DATA_DIR.exists()}, видео внутри: {has_videos(DATA_DIR)}")
if cv2 is None:
    print("opencv-python не установлен. Инвентаризация видео и кадры будут пропущены.")
    print("Установка: pip install opencv-python pandas numpy matplotlib seaborn tqdm")


## 1.1. Технические параметры видео

Для каждого ролика в `open/` собираются имя, FPS, число кадров, длительность в секундах и разрешение. Ожидаемый формат кейса — `.mov`. Поиск идёт рекурсивно: вложенные каталоги тоже учитываются.

Имена сопоставляются с разметкой по идентификатору `X-N`. Латинская и кириллическая «Х» считаются одним символом: в именах файлов они визуально совпадают.


In [ ]:
def clip_id_from_name(name: str) -> str:
    stem = Path(name).stem
    stem = stem.translate(str.maketrans({"Х": "X", "х": "x"}))
    match = re.search(r"(X-\d+)", stem, flags=re.IGNORECASE)
    if match:
        prefix, number = match.group(1).split("-")
        return f"X-{int(number)}"
    return stem


def clip_sort_key(clip_id: str):
    match = re.fullmatch(r"X-(\d+)", str(clip_id))
    if match:
        return int(match.group(1))
    return 10**9


def read_video_record(video_path: Path) -> dict:
    try:
        relative = str(video_path.resolve().relative_to(DATA_DIR.resolve()))
    except ValueError:
        relative = video_path.name
    record = {
        "video": video_path.name,
        "video_rel": relative,
        "clip_id": clip_id_from_name(video_path.name),
        "fps": np.nan,
        "frames": np.nan,
        "duration_sec": np.nan,
        "width": np.nan,
        "height": np.nan,
        "readable": False,
    }
    if cv2 is None:
        return record

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        print(f"Cannot open: {video_path.name}")
        return record

    fps = float(cap.get(cv2.CAP_PROP_FPS))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    record.update({
        "fps": fps,
        "frames": frame_count,
        "duration_sec": frame_count / fps if fps > 0 else 0.0,
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "readable": True,
    })
    cap.release()
    return record


def collect_videos(directory: Path) -> list[Path]:
    if not directory.exists():
        return []
    found = [
        path for path in directory.rglob("*")
        if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
    ]
    return sorted(found, key=lambda path: (clip_sort_key(clip_id_from_name(path.name)), path.name.lower()))


video_paths = collect_videos(DATA_DIR)

inventory = pd.DataFrame([read_video_record(path) for path in video_paths])
if inventory.empty:
    inventory = pd.DataFrame(columns=[
        "video", "video_rel", "clip_id", "fps", "frames", "duration_sec", "width", "height", "readable",
    ])
else:
    inventory = inventory.sort_values("clip_id", key=lambda s: s.map(clip_sort_key)).reset_index(drop=True)

inventory_path = PROCESSED_DIR / "video_inventory.csv"
inventory.to_csv(inventory_path, index=False)

print(f"Найдено видео: {len(inventory)}")
print(f"Успешно открыто: {int(inventory['readable'].sum()) if len(inventory) else 0}")
print(f"Сохранено: {inventory_path}")
inventory


## 1.2. Распределения длительности, FPS и разрешения

По ТЗ ролики должны лежать в диапазоне 10–145 секунд. Если файл короче или длиннее, его нужно проверить отдельно: сломанный контейнер, неверное число кадров или ролик не из обучающей выборки.


In [ ]:
def save_figure(fig, filename: str) -> Path:
    target = FIGURES_DIR / filename
    fig.tight_layout()
    fig.savefig(target, dpi=150, bbox_inches="tight")
    return target


readable = inventory[inventory["readable"] == True].copy() if len(inventory) else inventory

if readable.empty:
    print(f"В {DATA_DIR} нет читаемых видео. Положите ролики в папку open/ (или data/open/) и перезапустите ячейки 1.1–1.2.")
else:
    duration = readable["duration_sec"]
    outside = readable[(duration < 10) | (duration > 145)]
    print(
        "Длительность, сек: "
        f"min={duration.min():.1f}, median={duration.median():.1f}, "
        f"mean={duration.mean():.1f}, max={duration.max():.1f}"
    )
    print(f"Вне диапазона 10–145 сек: {len(outside)}")
    if len(outside):
        display(outside[["video", "duration_sec", "fps", "width", "height"]])

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(duration, bins=15, edgecolor="black", color="#4C78A8")
    ax.set_xlabel("Длительность видео, сек")
    ax.set_ylabel("Число видео")
    ax.set_title("Распределение длительности")
    save_figure(fig, "duration_distribution.png")
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 5))
    fps_counts = readable["fps"].round(2).value_counts().sort_index()
    fps_counts.plot(kind="bar", ax=ax, color="#F58518", edgecolor="black")
    ax.set_xlabel("FPS")
    ax.set_ylabel("Число видео")
    ax.set_title("Распределение частоты кадров")
    save_figure(fig, "fps_distribution.png")
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 5))
    resolution = readable["width"].astype(int).astype(str) + "x" + readable["height"].astype(int).astype(str)
    resolution.value_counts().plot(kind="bar", ax=ax, color="#54A24B", edgecolor="black")
    ax.set_xlabel("Разрешение")
    ax.set_ylabel("Число видео")
    ax.set_title("Распределение разрешений")
    save_figure(fig, "resolution_distribution.png")
    plt.show()


## 1.3. Фактическая разметка

Таблица ниже повторяет выданную классификацию 42 роликов. `source_label` не меняется из-за сомнения в правовой квалификации.

- `observed_event_type` описывает видимое действие и сохраняет неопределённость («предполагаемые», «в кадре не наблюдается»).
- `candidate_violation_type` — гипотеза о группе нарушения, а не доказанный проступок.
- `review_status` показывает, хватает ли публичной нормы или ещё нужны факты и внутренний регламент банка.
- `confidence` — качественная уверенность гипотезы до ручного просмотра: `medium` для более прямой визуальной формулировки, `low` если норма применяется только при дополнительном факте.
- Колонки `manual_observed_action` и `manual_confidence` заполняются после просмотра ролика. Пустое значение значит, что человек ещё не подтвердил событие.


In [ ]:
# clip_id, timecode, source_label_ru, observed_event, observed_event_type,
# candidate_violation_type, legal_basis, review_status
SUPPLIED_ROWS = [
    ("X-1", "00:22", "Нарушение", "После завершения пересчёта принятые деньги оставлены на рабочем столе", "accepted_cash_left_on_desk_after_count", "cash_storage", "№231, п. 20, абз. 2", "basis_identified"),
    ("X-2", "00:10", "Нарушение", "В кадре не наблюдается визуальная идентификация клиента, вносящего наличные", "cash_deposit_without_visible_id_check", "customer_identification", "№231, п. 19, пп. 1, если это клиент в смысле Правил", "conditional"),
    ("X-3", None, "Без нарушения", "Событие отдельно не описано в исходной разметке", "not_reported", None, None, "not_mapped"),
    ("X-4", "00:57", "Нарушение", "Деньги клиента №1 остались на столе при входе клиента №2", "prior_client_cash_left_during_next_client_entry", "cash_storage", "№231, п. 20, абз. 2; подтвердить, что деньги уже приняты", "basis_identified"),
    ("X-5", None, "Без нарушения", "Событие отдельно не описано в исходной разметке", "not_reported", None, None, "not_mapped"),
    ("X-8", None, "Без нарушения", "Событие отдельно не описано в исходной разметке", "not_reported", None, None, "not_mapped"),
    ("X-10", None, "Без нарушения", "Событие отдельно не описано в исходной разметке", "not_reported", None, None, "not_mapped"),
    ("X-11", "00:18", "Нарушение", "Ключ оставлен в замке сейфа", "safe_key_left_in_lock", "key_storage", "№231, п. 80 — отсылка к внутреннему регламенту банка", "internal_policy_required"),
    ("X-12", None, "Без нарушения", "Событие отдельно не описано в исходной разметке", "not_reported", None, None, "not_mapped"),
    ("X-13", "00:47", "Нарушение", "В видео не наблюдается регистрация суммы денег из инкассаторской сумки до помещения на хранение", "collection_cash_amount_registration_not_observed", "collection_cash_accounting", "№231, п. 24, пп. 2 — подтвердить по внутреннему порядку и учётным записям", "conditional"),
    ("X-14", None, "Без нарушения", "Событие отдельно не описано в исходной разметке", "not_reported", None, None, "not_mapped"),
    ("X-15", "00:15", "Нарушение", "Сумка оставлена без присмотра; содержимое не подтверждено", "unattended_bag_contents_unknown", "cash_storage", "№231, п. 20, абз. 2 — только если внутри ранее принятые клиентские деньги", "conditional"),
    ("X-16", None, "Без нарушения", "Событие отдельно не описано в исходной разметке", "not_reported", None, None, "not_mapped"),
    ("X-17", "00:07", "Нарушение", "Сумка с наличными принята; проверка личности и полномочий передавшего лица в кадре не наблюдается", "cash_bag_received_without_visible_identity_authority_check", "identity_and_authority_check", "№231, п. 19, пп. 1; Закон №191-IV, ст. 5 и ст. 7 — если лицо действует как представитель клиента", "conditional"),
    ("X-18", "00:20", "Нарушение", "В кадре не наблюдается визуальная идентификация клиента, вносящего наличные", "cash_deposit_without_visible_id_check", "customer_identification", "№231, п. 19, пп. 1, если это клиент в смысле Правил", "conditional"),
    ("X-19", None, "Без нарушения", "Событие отдельно не описано в исходной разметке", "not_reported", None, None, "not_mapped"),
    ("X-20", "00:05", "Нарушение", "Сейф открыт; кассир пересчитывает деньги из сейфа; контролирующий работник в кадре не наблюдается", "cash_count_from_safe_without_visible_controller", "cash_storage_or_recount_control", "№231, п. 20, абз. 2; п. 42; п. 80 — разные условия; подтвердить вид операции", "conditional"),
    ("X-21", None, "Без нарушения", "Событие отдельно не описано в исходной разметке", "not_reported", None, None, "not_mapped"),
    ("X-22", "00:14", "Нарушение", "Сотрудница отходит, ключ остаётся в замке сейфа", "employee_steps_away_key_left_in_safe_lock", "key_storage", "№231, п. 80 — отсылка к внутреннему регламенту банка", "internal_policy_required"),
    ("X-23", None, "Без нарушения", "Событие отдельно не описано в исходной разметке", "not_reported", None, None, "not_mapped"),
    ("X-24", "00:08", "Нарушение", "Предполагаемые банковские печати или штампы оставлены на столе без присмотра", "suspected_stamp_or_seal_left_unattended", "stamp_and_seal_storage", "№231, п. 80 — подтвердить, что это банковские печати или штампы", "internal_policy_required"),
    ("X-25", None, "Без нарушения", "Событие отдельно не описано в исходной разметке", "not_reported", None, None, "not_mapped"),
    ("X-26", "00:09", "Нарушение", "Сотрудница отходит, предполагаемые наличные остаются на рабочем столе", "employee_steps_away_suspected_cash_on_desk", "cash_storage_and_workstation_abandonment", "№231, п. 20, абз. 2; п. 22 — если обслуживание клиента ещё не завершено", "conditional"),
    ("X-27", None, "Без нарушения", "Событие отдельно не описано в исходной разметке", "not_reported", None, None, "not_mapped"),
    ("X-28", "00:07", "Нарушение", "Сотрудница отходит от ноутбука; состояние блокировки рабочей сессии не установлено", "employee_steps_away_workstation_lock_unknown", "information_security", "Внутренние правила ИБ ForteBank; точный пункт не установлен", "internal_policy_required"),
    ("X-29", None, "Без нарушения", "Событие отдельно не описано в исходной разметке", "not_reported", None, None, "not_mapped"),
    ("X-30", "00:13", "Нарушение", "Мужчина входит в рабочую кассовую зону; его полномочия не установлены", "person_enters_cash_work_area_authorization_unknown", "access_control", "Внутренний пропускной режим; Правила №14, п. 8 не подтверждают отсутствие полномочий", "internal_policy_required"),
    ("X-31", None, "Без нарушения", "Событие отдельно не описано в исходной разметке", "not_reported", None, None, "not_mapped"),
    ("X-32", "00:05", "Нарушение", "Сотрудница работает возле кассового оборудования при открытой двери", "cash_area_door_open_during_work", "cash_area_access_control", "Правила №14, п. 8; открытая дверь сама по себе не доказывает нарушение", "internal_policy_required"),
    ("X-33", "00:06", "Нарушение", "Сотрудница садится за рабочее место, входная дверь остаётся открытой", "employee_starts_work_entry_door_open", "cash_area_access_control", "Правила №14, п. 8; уточнить, какие помещения соединяет дверь", "internal_policy_required"),
    ("X-34", "00:13", "Нарушение", "Кассир пересчитывает принимаемые наличные после ухода клиента", "cash_count_after_client_departure", "cash_acceptance_without_client_present", "№231, п. 20, абз. 1 — если продолжается приём и пересчёт наличных клиента", "conditional"),
    ("X-35", None, "Без нарушения", "Событие отдельно не описано в исходной разметке", "not_reported", None, None, "not_mapped"),
    ("X-37", None, "Без нарушения", "Событие отдельно не описано в исходной разметке", "not_reported", None, None, "not_mapped"),
    ("X-38", "00:33", "Нарушение", "Клиент уходит; предполагаемые принятые деньги остаются на столе", "suspected_accepted_cash_left_after_client_departure", "cash_storage", "№231, п. 20, абз. 2 — подтвердить приём денег и завершение операции", "conditional"),
    ("X-39", None, "Без нарушения", "Событие отдельно не описано в исходной разметке", "not_reported", None, None, "not_mapped"),
    ("X-40", "00:17", "Нарушение", "Личная сумка и верхняя одежда размещены на кассовом столе", "employee_personal_items_on_cashier_desk", "personal_items_in_cash_unit", "№231, п. 12 — если это вещи сотрудника и место находится в кассовом узле", "basis_identified"),
    ("X-43", None, "Без нарушения", "Событие отдельно не описано в исходной разметке", "not_reported", None, None, "not_mapped"),
    ("X-44", "00:16", "Нарушение", "Сотрудница работает с предполагаемыми банкнотами без клиента", "employee_handles_suspected_cash_without_client", "cash_storage", "№231, п. 20, абз. 2 — только если это ранее принятые деньги и работа с ними завершена", "conditional"),
    ("X-45", None, "Без нарушения", "Событие отдельно не описано в исходной разметке", "not_reported", None, None, "not_mapped"),
    ("X-46", "00:11", "Нарушение", "На рабочем месте сотрудницы находится напиток рядом с предполагаемыми денежными средствами", "beverage_near_suspected_cash", "workplace_organization", "Внутренний регламент ForteBank; №231, п. 12 — только условная ссылка", "internal_policy_required"),
    ("X-47", "01:08", "Нарушение", "После приёма наличные не помещены в закрывающееся устройство хранения", "accepted_cash_not_moved_to_lockable_storage", "cash_storage", "№231, п. 20, абз. 2", "basis_identified"),
    ("X-48", None, "Без нарушения", "Событие отдельно не описано в исходной разметке", "not_reported", None, None, "not_mapped"),
]

CONFIDENCE_BY_STATUS = {
    "basis_identified": "medium",
    "conditional": "low",
    "internal_policy_required": "low",
    "not_mapped": None,
}


def parse_timecode(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan
    text = str(value).strip()
    if text in {"", "—", "-", "None", "nan"}:
        return np.nan
    if re.fullmatch(r"\d+(\.\d+)?", text):
        return float(text)
    parts = text.split(":")
    if not all(re.fullmatch(r"\d+(\.\d+)?", part) for part in parts):
        return np.nan
    seconds = 0.0
    for part in parts:
        seconds = seconds * 60 + float(part)
    return seconds


supplied = pd.DataFrame(SUPPLIED_ROWS, columns=[
    "clip_id", "timecode", "source_label_ru", "observed_event",
    "observed_event_type", "candidate_violation_type", "legal_basis", "review_status",
])
supplied["source_label"] = supplied["source_label_ru"].map({
    "Нарушение": "violation",
    "Без нарушения": "no_violation",
})
supplied["timecode_sec"] = supplied["timecode"].map(parse_timecode)
supplied["confidence"] = supplied["review_status"].map(CONFIDENCE_BY_STATUS)
supplied["manual_observed_action"] = pd.NA
supplied["manual_confidence"] = pd.NA

if EVENTS_PATH.exists():
    previous = pd.read_csv(EVENTS_PATH)
    manual_cols = [col for col in ("manual_observed_action", "manual_confidence") if col in previous.columns]
    if manual_cols and "clip_id" in previous.columns:
        supplied = supplied.drop(columns=manual_cols).merge(
            previous[["clip_id", *manual_cols]],
            on="clip_id",
            how="left",
        )
        filled = supplied["manual_observed_action"].notna().sum()
        print(f"Сохранены ручные пометки из {EVENTS_PATH.name}: {int(filled)}")

label_order = ["violation", "no_violation"]
events = supplied.sort_values("clip_id", key=lambda s: s.map(clip_sort_key)).reset_index(drop=True)
assert events["clip_id"].is_unique
assert set(events["source_label"]) <= set(label_order)

video_lookup = {}
if len(inventory):
    for row in inventory.itertuples(index=False):
        video_lookup.setdefault(row.clip_id, getattr(row, "video_rel", row.video))
events["video"] = events["clip_id"].map(video_lookup)

print(f"Роликов в разметке: {len(events)}")
print(events["source_label"].value_counts().to_string())
print(f"Роликов разметки без файла: {int(events['video'].isna().sum())}")
if len(inventory):
    extra = sorted(set(inventory["clip_id"]) - set(events["clip_id"]), key=clip_sort_key)
    print(f"Видео без строки разметки: {extra or 'нет'}")

events


## 1.2–1.3. Метки, категории и дисбаланс

Бинарная метка почти сбалансирована, но визуальные типы событий почти не повторяются. Для baseline это аргумент в пользу классификации `normal` / `violation`, а не отдельной модели на каждый тип. Повторяющиеся группы (`cash_storage`, идентификация, ключ в замке, открытая дверь) стоит проверять глазами: именно они могут дать устойчивый визуальный паттерн.


In [ ]:
label_counts = events["source_label"].value_counts().reindex(["violation", "no_violation"])
majority = int(label_counts.max())
minority = int(label_counts.min())
print(f"Доля violation: {label_counts['violation'] / len(events):.1%}")
print(f"Дисбаланс majority/minority: {majority / minority:.2f}")

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(
    x=label_counts.index,
    y=label_counts.values,
    hue=label_counts.index,
    ax=ax,
    palette=["#E45756", "#4C78A8"],
    legend=False,
)
ax.set_xlabel("Метка датасета")
ax.set_ylabel("Число роликов")
ax.set_title("Соотношение роликов с нарушением и без")
for index, value in enumerate(label_counts.values):
    ax.text(index, value + 0.3, str(int(value)), ha="center")
save_figure(fig, "label_distribution.png")
plt.show()

category_counts = (
    events["candidate_violation_type"]
    .dropna()
    .value_counts()
    .sort_values(ascending=True)
)
fig, ax = plt.subplots(figsize=(10, 6))
category_counts.plot(kind="barh", ax=ax, color="#72B7B2", edgecolor="black")
ax.set_xlabel("Число роликов")
ax.set_ylabel("Гипотеза категории")
ax.set_title("Кандидатные категории нарушений")
save_figure(fig, "candidate_violation_types.png")
plt.show()

type_counts = events.loc[events["source_label"].eq("violation"), "observed_event_type"].value_counts()
single_example = int((type_counts == 1).sum())
print(f"Уникальных визуальных типов среди нарушений: {type_counts.size}")
print(f"Типов с одним примером: {single_example}")
print("Повторяющиеся типы:")
print(type_counts[type_counts > 1].to_string() or "нет")

timed = events.dropna(subset=["timecode_sec"])
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(timed["timecode_sec"], bins=12, edgecolor="black", color="#B279A2")
ax.set_xlabel("Таймкод события, сек")
ax.set_ylabel("Число событий")
ax.set_title("Где в ролике отмечены нарушения")
save_figure(fig, "timecode_distribution.png")
plt.show()
print(
    "Таймкоды нарушений, сек: "
    f"min={timed['timecode_sec'].min():.0f}, "
    f"median={timed['timecode_sec'].median():.0f}, "
    f"max={timed['timecode_sec'].max():.0f}"
)


## 1.4. Ручной просмотр нарушений

Для каждого отмеченного события нужно своими словами записать видимое действие и уверенность. Готовая формулировка в `observed_event` — стартовая гипотеза. После просмотра её записывают в `manual_observed_action`, а уверенность — в `manual_confidence` (`high`, `medium` или `low`).

Нормативную ссылку не стоит принимать автоматически. В частности:

- отсутствие документа в кадре не доказывает, что идентификация не проводилась;
- открытая дверь не доказывает нарушение изоляции кассового узла;
- «предполагаемые наличные» остаются предполагаемыми, пока банкноты не различимы;
- регистрация суммы и полномочия входящего человека по одному кадру не устанавливаются.

Ниже таблица только для роликов с меткой `violation`.


In [ ]:
review_columns = [
    "clip_id", "video", "timecode", "timecode_sec", "observed_event",
    "observed_event_type", "candidate_violation_type", "legal_basis",
    "review_status", "confidence", "manual_observed_action", "manual_confidence",
]
review = events.loc[events["source_label"].eq("violation"), review_columns].reset_index(drop=True)
print(f"Событий для ручного просмотра: {len(review)}")
review


## 1.5. Кадры вокруг таймкода

Для каждого нарушения сохраняется последовательность: 4 секунды до отметки, сама отметка и 4 секунды после, с шагом 2 секунды. Так видно действие, а не один удачный кадр.

В ноутбуке показывается первый ролик, для которого файл найден. Контактные листы остальных сохраняются в `reports/figures/event_frames/`.


In [ ]:
def read_frame(video_path, time_sec):
    if cv2 is None:
        return None
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        return None
    cap.set(cv2.CAP_PROP_POS_MSEC, float(time_sec) * 1000)
    ok, frame = cap.read()
    cap.release()
    if not ok or frame is None:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def window_timestamps(center_sec, window=WINDOW_SEC, step=FRAME_STEP_SEC):
    start = max(0.0, float(center_sec) - window)
    stop = float(center_sec) + window
    stamps = np.arange(start, stop + 1e-6, step)
    return [round(float(stamp), 2) for stamp in stamps]


def resolve_video(clip_id: str, video_name):
    if isinstance(video_name, str) and video_name:
        for candidate in (DATA_DIR / video_name, Path(video_name)):
            if candidate.is_file():
                return candidate
    if not DATA_DIR.exists():
        return None
    for path in DATA_DIR.rglob("*"):
        if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS:
            if clip_id_from_name(path.name) == clip_id:
                return path
    return None


def save_event_strip(clip_id: str, video_path: Path, center_sec: float) -> Path | None:
    stamps = window_timestamps(center_sec)
    frames = [read_frame(video_path, stamp) for stamp in stamps]
    if all(frame is None for frame in frames):
        print(f"{clip_id}: кадры не прочитаны ({video_path.name}, t={center_sec})")
        return None

    fig, axes = plt.subplots(1, len(stamps), figsize=(3.2 * len(stamps), 3.6))
    if len(stamps) == 1:
        axes = [axes]
    for ax, stamp, frame in zip(axes, stamps, frames):
        if frame is not None:
            ax.imshow(frame)
        ax.set_title(f"{stamp:.0f}s" if float(stamp).is_integer() else f"{stamp:.1f}s")
        ax.axis("off")
    fig.suptitle(f"{clip_id} — окно вокруг {center_sec:.0f}s", y=1.02)
    target = EVENT_FRAMES_DIR / f"{clip_id}.png"
    fig.savefig(target, dpi=120, bbox_inches="tight")
    return target


saved_strips = []
shown = False
violations = events.dropna(subset=["timecode_sec"])

if cv2 is None or not video_paths:
    print(f"Кадры не извлечены: нет opencv или нет файлов в {DATA_DIR}.")
else:
    for row in tqdm(list(violations.itertuples(index=False)), desc="event frames"):
        video_path = resolve_video(row.clip_id, row.video)
        if video_path is None:
            continue
        target = save_event_strip(row.clip_id, video_path, row.timecode_sec)
        if target is None:
            continue
        saved_strips.append(target)
        if not shown:
            plt.show()
            shown = True
        else:
            plt.close()

print(f"Сохранено последовательностей кадров: {len(saved_strips)}")


## 1.6. Зоны кадра

Зоны имеют смысл только там, где они реально различимы. Для кассовых роликов обычно проверяют рабочую область кассира, место контакта с клиентом, место наличных, сейф или иное хранение и вход. Координаты задаются в пикселях `(x1, y1, x2, y2)` после просмотра кадра. Пустой словарь означает, что зоны ещё не размечены.


In [ ]:
# Пример после просмотра кадра:
# ZONES = {
#     "X-1": {
#         "cashier_work_area": (80, 60, 900, 700),
#         "cash_placement": (400, 350, 700, 560),
#     },
# }
ZONES: dict[str, dict[str, tuple[int, int, int, int]]] = {}

ZONE_COLORS = ["#E45756", "#4C78A8", "#54A24B", "#F58518", "#B279A2", "#72B7B2"]


def draw_zones(frame, zones: dict, title: str = ""):
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.imshow(frame)
    for index, (name, box) in enumerate(zones.items()):
        x1, y1, x2, y2 = box
        color = ZONE_COLORS[index % len(ZONE_COLORS)]
        rect = Rectangle(
            (x1, y1), max(x2 - x1, 1), max(y2 - y1, 1),
            fill=False, edgecolor=color, linewidth=2,
        )
        ax.add_patch(rect)
        ax.text(x1, max(y1 - 8, 8), name, color=color, fontsize=10, backgroundcolor="white")
    ax.set_title(title)
    ax.axis("off")
    return fig


if ZONES and cv2 is not None:
    for clip_id, zones in ZONES.items():
        row = events.loc[events["clip_id"].eq(clip_id)]
        if row.empty or pd.isna(row.iloc[0]["timecode_sec"]):
            print(f"{clip_id}: нет таймкода для отрисовки зон")
            continue
        video_path = resolve_video(clip_id, row.iloc[0]["video"])
        if video_path is None:
            print(f"{clip_id}: видео не найдено")
            continue
        frame = read_frame(video_path, row.iloc[0]["timecode_sec"])
        if frame is None:
            print(f"{clip_id}: кадр не прочитан")
            continue
        fig = draw_zones(frame, zones, title=f"{clip_id} @ {row.iloc[0]['timecode_sec']:.0f}s")
        save_figure(fig, f"zones_{clip_id}.png")
        plt.show()
else:
    print("Зоны не заданы. После просмотра кадра добавьте прямоугольники в ZONES и перезапустите ячейку.")


## 1.7. Экспорт `events.csv`

Файл соединяет исходную метку, таймкод, визуальный тип, гипотезу категории, правовую оговорку и ручные поля. Технические параметры видео подмешиваются, когда ролик уже прочитан. Повторный запуск не затирает `manual_observed_action` и `manual_confidence`.


In [ ]:
export = events.copy()
if len(inventory):
    keep_cols = [c for c in ["clip_id", "video_rel", "fps", "frames", "duration_sec", "width", "height", "readable"] if c in inventory.columns]
    technical = inventory[keep_cols].rename(columns={"video_rel": "video_file"})
    export = export.merge(technical, on="clip_id", how="left")

column_order = [
    "clip_id", "video", "video_file", "source_label", "source_label_ru",
    "timecode", "timecode_sec", "observed_event", "observed_event_type",
    "candidate_violation_type", "legal_basis", "review_status", "confidence",
    "manual_observed_action", "manual_confidence",
    "fps", "frames", "duration_sec", "width", "height", "readable",
]
export = export[[col for col in column_order if col in export.columns]]
export.to_csv(EVENTS_PATH, index=False)
print(f"Сохранено: {EVENTS_PATH} ({len(export)} строк)")

checklist = {
    "Проверить все видео и их параметры": bool(len(readable)) and int(readable["readable"].sum()) == len(events),
    "Построить распределение длительности, FPS и разрешений": (FIGURES_DIR / "duration_distribution.png").exists(),
    "Загрузить фактическую разметку": len(events) == 42 and events["source_label"].nunique() == 2,
    "Посмотреть все отмеченные нарушения": int(events["manual_observed_action"].notna().sum()) >= int(label_counts["violation"]),
    "Составить категории предполагаемых событий": events.loc[events["source_label"].eq("violation"), "candidate_violation_type"].notna().all(),
    "Сохранить кадры до и после событий": len(saved_strips) >= int(label_counts["violation"]),
    "Подготовить таблицу events.csv": EVENTS_PATH.exists() and len(export) == 42,
}
status = pd.DataFrame({
    "Задача": list(checklist),
    "Готово": ["да" if done else "ещё нет" for done in checklist.values()],
})
print(f"{sum(checklist.values())} из {len(checklist)} выполнено")
status


## Что следует из разметки до обучения

Пока ролики не просмотрены вручную, для моделирования достаточно четырёх выводов.

1. Учить сразу многоклассовый детектор типов рано: почти каждый визуальный тип встретился один раз.
2. Первый baseline — бинарное `no_violation` / `violation`. Валидацию нужно резать по целым роликам, иначе соседние кадры одного видео попадут и в train, и в validation.
3. Крупнейшая группа гипотез — хранение и оставление ценностей на столе. Если на кадрах повторяется один и тот же жест «отошёл и оставил предмет», для него полезны короткий видеоклассификатор и зоны стола. Если событие — это пересечение двери или передача предмета, тогда имеет смысл YOLO и трекинг.
4. Часть меток зависит от фактов, которых нет в кадре: статус клиента, факт приёмки денег, полномочия, внутренний регламент. Модель должна предсказывать наблюдаемое событие. Юридическую формулировку оставляет человек.

Следующая тетрадка — `02_model_experiments.ipynb`. Её имеет смысл открывать после того, как в `events.csv` заполнены ручные описания и в `open/` лежат все ролики открытой выборки.
